# EDA

Смотрим на данные перед обучением: размер, пропуски, баланс классов, корреляции.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# читаем данные
df = pd.read_csv('../data/UCI_Credit_Card.csv')
df.shape

In [ ]:
# первые строки
df.head()

In [ ]:
# типы и пропуски
df.info()

In [ ]:
# пропусков нет, данные уже чистые
df.isnull().sum().sum()

## Баланс классов

In [ ]:
# смотрим целевую переменную — есть ли дисбаланс
target = 'default.payment.next.month'
df[target].value_counts(normalize=True)

In [ ]:
# дефолтов ~22%, классы несбалансированы — нужен class_weight='balanced' в моделях
sns.countplot(x=target, data=df)
plt.title('Распределение целевой переменной')
plt.show()

## Демография

In [ ]:
# пол: 1=муж, 2=жен
df.groupby('SEX')[target].mean()

In [ ]:
# образование: 1=магистратура, 2=университет, 3=школа
df.groupby('EDUCATION')[target].mean()

In [ ]:
# возраст
df['AGE'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['AGE'], bins=30, ax=axes[0])
axes[0].set_title('Распределение возраста')
sns.boxplot(x=target, y='AGE', data=df, ax=axes[1])
axes[1].set_title('Возраст vs дефолт')
plt.tight_layout()
plt.show()

## Кредитный лимит

In [ ]:
# клиенты с меньшим лимитом чаще уходят в дефолт
sns.boxplot(x=target, y='LIMIT_BAL', data=df)
plt.title('Кредитный лимит vs дефолт')
plt.yscale('log')
plt.show()

## Корреляции

In [ ]:
# считаем корреляцию признаков с таргетом
corr = df.drop(columns=['ID']).corr()[target].sort_values(ascending=False)
corr

In [ ]:
# сильнее всего с дефолтом связаны PAY_0..PAY_6 — история просрочек
# это ожидаемо: если клиент уже просрочил платежи, он скорее уйдёт в дефолт
plt.figure(figsize=(10, 8))
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', target]
sns.heatmap(df[pay_cols].corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Корреляция PAY_* с таргетом')
plt.show()

## Выводы

- Данных 30000 строк, 25 колонок, пропусков нет
- Классы несбалансированы (~22% дефолтов) — нужен class_weight='balanced'
- Сильнейшие предикторы — PAY_0..PAY_6 (история просрочек)
- Кредитный лимит и возраст слабо коррелируют с таргетом
- Доп. предобработка не требуется: все признаки числовые, масштабирование только для LogReg